In [2]:
!pip install transformers

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 3.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.4/803.4 kB 2.3 MB/s eta 0:00:00-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 2.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 2.0 MB/s eta 0:00:00a 0:00:01


In [4]:
!pip install peft

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [6]:
!pip install evaluate

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 956.8 kB/s eta 0:00:0000:01:02
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 19.0.1
    Uninstalling pyarrow-19.0.1:
      Successfully uninstalled pyarrow-19.0.1


In [31]:
!pip install rouge_score
!pip install nltk

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 4.3 MB/s eta 0:00:00a 0:00:01
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24987 sha256=5e10e63c7b4dfe944188069dbd0e2aacf3d70b23470f35d611c64cc5392f16a1
  Stored in directory: /tmp/pip-ephem-wheel-cache-7cqw_w3u/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [2]:
pip install --upgrade peft

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import os
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from peft import LoraConfig, get_peft_model
from transformers import get_cosine_schedule_with_warmup
from tqdm.auto import tqdm
import evaluate

2025-11-01 08:58:28.023298: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-01 08:58:29.735669: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761987510.364528     552 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761987510.549542     552 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-11-01 08:58:31.962294: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
df = pd.read_csv("Artemis/paintings_cleaned.csv")
df

,art_style,painting,emotion,utterance,repetition
0,Art_Nouveau_Modern,vasily-polenov_details-of-golden-gates-1893,anger,"I love this mosaic work! However, it is unfini...",7
1,Realism,klavdy-lebedev_svyatoslav-s-meeting-with-emper...,anger,The people have very stern looks on their face...,6
2,Northern_Renaissance,albrecht-durer_pages-of-marginal-drawings-for-...,anger,This is a page in a book written in a language...,7
3,Northern_Renaissance,rogier-van-der-weyden_saint-jerome-and-the-lio...,anger,The look in the priest's eyes while he is hold...,6
4,Romanticism,john-william-waterhouse_saint-eulalia-1885,anger,It looks like a tragic scene where it looks li...,7
...,...,...,...,...,...
4035,Action_painting,jackson-pollock_number-17-1949,anger,The colors and lines all clash with each other...,58
4036,New_Realism,john-french-sloan_gray-and-brass-1907,anger,"The elite, rich white people riding through to...",49
4037,Cubism,gino-severini_a-dancer-1,anger,The man dances like Jazz plays on the radio.,48
4038,Cubism,gino-severini_a-dancer-1,anger,Too many colors and shapes all combined in a s...,48


In [3]:
folder_path = "Artemis/wikiart_images"

paths = os.listdir(folder_path)
image_paths = []
for i in paths:
    image_paths.append(i.replace(".jpg",""))


In [4]:
len(image_paths)

3528

In [5]:
df.drop_duplicates(subset=["painting"],inplace=True)

In [6]:
df = df[df["painting"].isin(image_paths)]

In [7]:
df = df[df["painting"] != "n.c.-wyeth_black-spot"]

In [8]:
x_train, x_val, y_train, y_val = train_test_split(
    df["painting"], df["utterance"],
    test_size=0.14,      
    random_state=42,     
    shuffle=True,                   
)
x_test, x_val, y_test, y_val = train_test_split(
    x_val, y_val,
    test_size=0.5,        
    random_state=42,      
    shuffle=True,                   
)

In [9]:
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed",cache_dir="./model")
tokenizer = processor.tokenizer
image_processor = processor.image_processor

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [10]:
train_image_paths = x_train.to_numpy()
train_captions = y_train.to_numpy()
val_image_paths = x_val.to_numpy()
val_captions = y_val.to_numpy()
test_image_paths = x_test.to_numpy()
test_captions = y_test.to_numpy()

In [11]:
class CaptionPromptDataset(Dataset):
    def __init__(self, image_paths, captions, image_processor, tokenizer, folder_path, max_length=528):
        self.image_paths = image_paths
        self.captions = captions
        self.image_processor = image_processor
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.folder_path = folder_path

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(f"{folder_path}/{self.image_paths[idx]}.jpg").convert("RGB")
        pixel_values = self.image_processor(images=image, return_tensors="pt").pixel_values.squeeze(0)

        answer = self.captions[idx] + self.tokenizer.eos_token

        ans_ids = self.tokenizer(
            answer,
            add_special_tokens=False,
            truncation=False,
            max_length=self.max_length,
            return_tensors="pt"
        ).input_ids[0]

        return pixel_values, ans_ids


In [12]:
import torch.nn.functional as F

def collate(batch):
    pixel_values, ans_ids = zip(*batch)

    pixel_values = torch.stack(pixel_values)
    ans_ids = torch.nn.utils.rnn.pad_sequence(ans_ids, batch_first=True, padding_value=-100) 
    
    return pixel_values, ans_ids

In [13]:
train_ds = CaptionPromptDataset(image_paths=train_image_paths,captions=train_captions,image_processor=image_processor,tokenizer=tokenizer,folder_path=folder_path)
val_ds = CaptionPromptDataset(image_paths=val_image_paths,captions=val_captions,image_processor=image_processor,tokenizer=tokenizer,folder_path=folder_path)

In [14]:
train = DataLoader(
        train_ds, batch_size=10, shuffle=False,
        collate_fn=collate,
)

val = DataLoader(
        val_ds, batch_size=10, shuffle=False,
        collate_fn=collate,
)

In [15]:
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-printed", cache_dir="./model")

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
model  = VisionEncoderDecoderModel.from_pretrained("best_trocr_model")

In [16]:
model

VisionEncoderDecoderModel(
  (encoder): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=False)
              (key): Linear(in_features=768, out_features=768, bias=False)
              (value): Linear(in_features=768, out_features=768, bias=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (i

In [17]:
model.config.decoder_start_token_id = processor.tokenizer.bos_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.eos_token_id

In [18]:
for param in model.encoder.parameters():
    param.requires_grad = False

lora_config = LoraConfig(
    r=8,  # Use r=8 as you wanted for decoder
    lora_alpha=16,
    target_modules=[
        "query", "key", "value", 
      #  "self_attn.out_proj", "encoder_attn.k_proj", "encoder_attn.v_proj", 
       # "encoder_attn.q_proj", "encoder_attn.out_proj", "fc1", "fc2"
    ],
    lora_dropout=0.3,
    bias="none",
)

model = get_peft_model(model, lora_config)

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
device

device(type='cuda')

In [20]:
num_epochs = 10
warmup_ratio = 0.1
total_steps = len(train) * num_epochs
warmup_steps = int(total_steps * warmup_ratio)

In [21]:
optimizer = AdamW(model.parameters(), lr=5e-6, weight_decay=0.01)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
    num_cycles=0.45
)

In [22]:
bleu = evaluate.load("bleu",cache_dir="./metrics", device=device)
rouge = evaluate.load("rouge",cache_dir="./metrics", device=device)

best_bleu = 0.0
best_rouge = 0.0
save_path = "best_trocr_model"

In [23]:
from torch.nn import functional as F
from tqdm import tqdm
import copy

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    loop = tqdm(train, desc=f"Epoch {epoch+1}/{num_epochs} [Training]", leave=False)

    for pixel_values, labels in loop:
        pixel_values = pixel_values.to(device)
        labels = labels.to(device)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_train_loss = total_loss / len(train)
    print(f"Epoch {epoch+1}: Train loss = {avg_train_loss:.4f}")

    # ---- validation ----
    model.eval()
    val_loss = 0.0
    preds, refs = [], []

    with torch.no_grad():
        loop_val = tqdm(val, desc=f"Epoch {epoch+1}/{num_epochs} [Validation]", leave=False)
        for pixel_values, labels in loop_val:
            pixel_values = pixel_values.to(device)
            labels = labels.to(device)

            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss
            val_loss += loss.item()

            generated_ids = model.generate(pixel_values, max_length=64, num_beams=4)
            decoded_preds = processor.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
            labels_for_decoding = labels.clone()
            labels_for_decoding[labels_for_decoding == -100] = processor.tokenizer.pad_token_id
            decoded_labels = processor.tokenizer.batch_decode(labels_for_decoding, skip_special_tokens=True)

            preds.extend(decoded_preds)
            refs.extend([[l] for l in decoded_labels])

    avg_val_loss = val_loss / len(val)
    bleu_score = bleu.compute(predictions=preds, references=refs)["bleu"]
    rouge_score = rouge.compute(predictions=preds, references=[r[0] for r in refs])["rougeL"]

    print(f"Epoch {epoch+1}: Val loss = {avg_val_loss:.4f} | BLEU = {bleu_score:.4f} | ROUGE-L = {rouge_score:.4f}")

    if rouge_score > best_rouge:
       best_rouge = rouge_score

       temp_model = copy.deepcopy(model)
       merged = temp_model.merge_and_unload()

       merged.save_pretrained(save_path)
       processor.save_pretrained(save_path)

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
/opt/conda/lib/python3.12/site-packages/PIL/Image.py:3402: DecompressionBombWarning: Image size (99962094 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
                                                                       

Epoch 1: Train loss = 7.8936


Epoch 1: Val loss = 7.8487 | BLEU = 0.0000 | ROUGE-L = 0.0456


Epoch 2: Train loss = 7.8427


Epoch 2: Val loss = 7.7558 | BLEU = 0.0000 | ROUGE-L = 0.0565


...

Epoch 8: Val loss = 7.4139 | BLEU = 0.0000 | ROUGE-L = 0.0767


Epoch 9: Train loss = 7.4778


Epoch 9: Val loss = 7.4059 | BLEU = 0.0000 | ROUGE-L = 0.0775


Epoch 10: Train loss = 7.4708


Epoch 10: Val loss = 7.4030 | BLEU = 0.0000 | ROUGE-L = 0.0768


In [24]:
pixel_values, labels = next(iter(train))
pixel_values = pixel_values.to(device)
labels = labels.to(device)

outputs = model(pixel_values=pixel_values, labels=labels)
loss = outputs.loss 
loss.backward()

In [34]:
import copy
temp_model = copy.deepcopy(model)
temp_model.merge_and_unload()
temp_model

PeftModel(
  (base_model): LoraModel(
    (model): VisionEncoderDecoderModel(
      (encoder): ViTModel(
        (embeddings): ViTEmbeddings(
          (patch_embeddings): ViTPatchEmbeddings(
            (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
          )
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (encoder): ViTEncoder(
          (layer): ModuleList(
            (0-11): 12 x ViTLayer(
              (attention): ViTAttention(
                (attention): ViTSelfAttention(
                  (query): Linear(in_features=768, out_features=768, bias=False)
                  (key): Linear(in_features=768, out_features=768, bias=False)
                  (value): Linear(in_features=768, out_features=768, bias=False)
                )
                (output): ViTSelfOutput(
                  (dense): Linear(in_features=768, out_features=768, bias=True)
                  (dropout): Dropout(p=0.0, inplace=False)
                )
             

In [24]:
model.merge_and_unload()
model.save_pretrained(save_path)
processor.save_pretrained(save_path)

[]

In [ ]:
model2  = VisionEncoderDecoderModel.from_pretrained("best_trocr_model_encoder_LoRA").to(device)

In [27]:
generated_ids = model2.generate(pixel_values)
processor.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

['  is it it it it it it it it it it it it it it it it it it']

In [31]:
torch.save(optimizer.state_dict(), f"{save_path}/optimizer.pt")
torch.save(scheduler.state_dict(), f"{save_path}/scheduler.pt")